In [1]:
import json
import os
from typing import Optional

import numpy as np
import pandas as pd
import torch

from direction_learning import DirVectors
from logging_setup import create_logger
from phi_3_5_constants import dsets_index_path, token_lengths_path, train_split_records_path, \
    validation_split_records_path, probes_folder, dsets_folder, finalized_activations_dir, hidden_state_size, \
    directions_results_folder
from phi_3_5_probe import ProbesForDataset, train_probes_for_dset

In [2]:
logger = create_logger(__name__)

In [3]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")
num_dsets = dsets_index_df.shape[0]

In [4]:
with token_lengths_path.open("r") as f:
    record_lengths_in_tokens = json.load(f)
with train_split_records_path.open("r") as f:
    train_split_record_idxs = json.load(f)
with validation_split_records_path.open("r") as f:
    validation_split_record_idxs = json.load(f)

In [5]:
probes_folder.mkdir(exist_ok=True)

In [6]:
all_dsets_activations: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]
all_dsets_labels: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]


In [7]:
all_dsets_probes: list[Optional[ProbesForDataset]] = [None for _ in range(num_dsets)]

In [ ]:
for dset_idx, dset_dtls in dsets_index_df.iterrows():
    categ_nm = dset_dtls["Categ_Folder"]
    dset_file_nm = dset_dtls["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    
    dataset = pd.read_csv(dsets_folder / categ_nm / dset_file_nm)
    dset_size = dataset.shape[0]
    dset_labels = torch.from_numpy(dataset['label'].to_numpy().astype(np.float32)[:, np.newaxis])
    all_dsets_labels[dset_idx] = dset_labels
    
    activs_path = finalized_activations_dir / categ_nm / (dset_nm + ".pt")
    relevant_activations = torch.load(activs_path, weights_only=True)
    assert relevant_activations.shape == (2, dset_size, hidden_state_size)
    all_dsets_activations[dset_idx] = relevant_activations
    
    train_split_activs = relevant_activations[:, train_split_record_idxs[str(dset_idx)], :]
    train_split_truth_labels = dset_labels[train_split_record_idxs[str(dset_idx)], :]
    
    val_split_activs = relevant_activations[:, validation_split_record_idxs[str(dset_idx)], :]
    val_split_truth_labels = dset_labels[validation_split_record_idxs[str(dset_idx)], :]
    
    dirs_for_dset_path = directions_results_folder / categ_nm / (dset_nm + ".pt")
    if not dirs_for_dset_path.exists():
        logger.warning(f"Directions for {dset_nm} not found, stopping (presuming that we've reached the end of the set of datasets whose directions have been computed)")
        break
    dirs_dict = torch.load(dirs_for_dset_path, weights_only=True)
    assert isinstance(dirs_dict, dict)
    mean_truth_polarity_dirs_for_dset = DirVectors(**dirs_dict)
    
    train_probes_for_dset(probes_folder / categ_nm, dset_nm, train_split_activs, train_split_truth_labels,
                          val_split_activs, val_split_truth_labels, mean_truth_polarity_dirs_for_dset)

2024-10-26 10:13:53,329;phi_3_5_probe;INFO:skipping the training of the layer18 probe for 131 records of data animal_class in the location trained_probes\animal_class because the file trained_probes\animal_class\animal_class_lyr18_probe.pth already exists
2024-10-26 10:13:53,345;phi_3_5_probe;INFO:skipping the training of the layer25 probe for 131 records of data animal_class in the location trained_probes\animal_class because the file trained_probes\animal_class\animal_class_lyr25_probe.pth already exists
2024-10-26 10:13:53,345;phi_3_5_probe;INFO:skipping the training of the layers18 and 25 probe for 131 records of data animal_class in the location trained_probes\animal_class because the file trained_probes\animal_class\animal_class_lyrs18_and_25_probe.pth already exists
2024-10-26 10:13:53,392;phi_3_5_probe;INFO:skipping the training of the layer18 probe for 400 records of data animal_class_conj in the location trained_probes\animal_class because the file trained_probes\animal_class

In [ ]:
#TODO write code for probe training for multi-dataset scenarios

In [9]:
# TODO check for relationship between final 'cost' of OLS process for a given dataset-scenario and layer choice (after normalizing for number of records) and the best validation loss that can be achieved by a probe using the given directions on the given dataset-scenario